# 数独

## 2020年4月11日 23:08:12(未完成)

- [x] 数独 
- [ ] 质数分解
- [ ] 编辑距离
- [ ] 滑动窗口
- [ ] 一元一次方程组
- [ ] 最长无重复子串
- [ ] 最长回文子串


> 大概两小时,独立完成数独求解算法  
> 现在速度确实是慢
> 星期六, 2020年4月11日 23:08:12

## 数独

`9*9`数独
- 判断当前输入是否有效
- 求解数独

算法思路说明:

(1) 使用三个量来记录数独情况:
- rows 标记每一行中当前数字出现情况`rows[0][2]=1`表示第一行存在一个3 (2+1索引从0开始)
- cols 标记每一列中当前数字出现情况`cols[0][5]=1`表示第一列存在一个6
- cells 标记每一个正方形中当前数字出现情况`cells[0][8]`表示第一个三角中存在一个9

(2) 对数独进行一次遍历可以获取数独的初始情况

(3) 简单将`rows`,`cols`,`cells`按照对应数字相加,表示数字的优先级  
优先级越高越早计算,对算法越有利

(4) 按照(3)中优先顺序处理,使用回溯法递归求解

递归中的一个关键步骤是当前可能值得求解,每次往数独中填数时都会改变统计量 rows,cols,cells

每次处理固定位置row,col时,都可以根据:
- `rows[row]` row 行出现过哪些数字
- `cols[col]` col 行出现过哪些数字
- `cells[map2cell(row,col)]` (row,col)cell单元出现过哪些数字

推算出row,col上允许出现得数字(排除上述出现得数字即可)

> 这里使用的是全局变量,也是引用变量,每一次递归调用的变动都会对全局造成影响,需要手动恢复.


In [1]:
# def check_sudoku(sudoku):
sudoku = [
  ["5","3",".",".","7",".",".",".","."],
  ["6",".",".","1","9","5",".",".","."],
  [".","9","8",".",".",".",".","6","."],
  ["8",".",".",".","6",".",".",".","3"],
  ["4",".",".","8",".","3",".",".","1"],
  ["7",".",".",".","2",".",".",".","6"],
  [".","6",".",".",".",".","2","8","."],
  [".",".",".","4","1","9",".",".","5"],
  [".",".",".",".","8",".",".","7","9"]
]

cols  = [[0]*9 for _ in range(9)]
rows  = [[0]*9 for _ in range(9)]
cells = [[0]*9 for _ in range(9)]

#      0-2 3-5 6-8
# 0-2 | 0   1   2
# 3-5 | 3   4   5 
# 6-8 | 6   7   8
def map2cell(row,col):
    if 0 <= row <= 2:  
        if 0 <= col <= 2: return 0
        elif 3 <= col <= 5: return 1
        elif 6 <= col <= 8: return 2
    elif 3 <= row <= 5:
        if 0 <= col <= 2: return 3
        elif 3 <= col <= 5: return 4
        elif 6 <= col <= 8: return 5
    elif 6 <= row <= 8:
        if 0 <= col <= 2: return 6
        elif 3 <= col <= 5: return 7
        elif 6 <= col <= 8: return 8

# 生成统计结果
for row in range(9):
    for col in range(9):
        cur = sudoku[row][col]
        if cur != ".":
            cur_idx = int(cur)-1
            cols[col][cur_idx] +=1
            rows[row][cur_idx] +=1 
            cells[map2cell(row,col)][cur_idx] +=1

# 判断是否有效 
has_bug = True in [cell >=2  for i in range(9) for cell in cells[:][i]] or \
    True in [ col >=2  for i in range(9) for col in cols[:][i] ] or \
    True in [ row >=2  for i in range(9) for row in rows[:][i] ] 

# 使用热值来生成一个计算顺序
def heat_compute(row,col):
    heat = sum(rows[row])+sum(cols[col])+sum(cells[map2cell(row,col)])
    return heat

# 对于热值高的点优先计算可以减少时间复杂度
point_count = [(heat_compute(row,col),row,col) for row in range(9) for col in range(9) if sudoku[row][col] == '.' ] 
point4solve = sorted(point_count,reverse=True)

def sudoku_solver(idx):   
    if idx < 0:
        return False
    if idx == len(point4solve):
        return True
    _,row,col = point4solve[idx]
    possible_val = []
    for v in range(9):
        if rows[row][v] == 0 and cols[col][v] == 0 and cells[map2cell(row,col)][v] == 0:
            possible_val.append(v)    
    for val in possible_val:
        sudoku[row][col] = str(val+1)
        cols[col][val] += 1
        rows[row][val] += 1
        cells[map2cell(row,col)][val] +=1
        if sudoku_solver(idx+1): # 递归 
            return True
        cols[col][val] -= 1
        rows[row][val] -= 1
        cells[map2cell(row,col)][val] -=1
        sudoku[row][col] = '.'
    return False

sudoku_solver(0)

True

In [2]:
sudoku

[['5', '3', '4', '6', '7', '8', '9', '1', '2'],
 ['6', '7', '2', '1', '9', '5', '3', '4', '8'],
 ['1', '9', '8', '3', '4', '2', '5', '6', '7'],
 ['8', '5', '9', '7', '6', '1', '4', '2', '3'],
 ['4', '2', '6', '8', '5', '3', '7', '9', '1'],
 ['7', '1', '3', '9', '2', '4', '8', '5', '6'],
 ['9', '6', '1', '5', '3', '7', '2', '8', '4'],
 ['2', '8', '7', '4', '1', '9', '6', '3', '5'],
 ['3', '4', '5', '2', '8', '6', '1', '7', '9']]

### 实验中发现的一个问题

如下例,这结果好像 or 和 and 反过来了一样,有点显然违背常识(卡了我很久)

todo 目前没有解答 

In [3]:
x = [1,0,1]
y = [1,1,1]
print("x\t",x)
print("y\t",y)
print("x"," or","y\t",x or y)
print("x","and","y\t",x and y)

x	 [1, 0, 1]
y	 [1, 1, 1]
x  or y	 [1, 0, 1]
x and y	 [1, 1, 1]


## [LeetCode 37. 解数独](https://leetcode-cn.com/problems/sudoku-solver/)


AC 66% 50%

In [4]:
class Solution(object):
    
    def map2cell(self,row,col):
        if 0 <= row <= 2:  
            if 0 <= col <= 2: return 0
            elif 3 <= col <= 5: return 1
            elif 6 <= col <= 8: return 2
        elif 3 <= row <= 5:
            if 0 <= col <= 2: return 3
            elif 3 <= col <= 5: return 4
            elif 6 <= col <= 8: return 5
        elif 6 <= row <= 8:
            if 0 <= col <= 2: return 6
            elif 3 <= col <= 5: return 7
            elif 6 <= col <= 8: return 8
    
    def heat_compute(self,row,col):
        heat = sum(self.rows[row]) + \
            sum(self.cols[col]) + \
            sum(self.cells[self.map2cell(row,col)])
        return heat

    def sudoku_solver(self,idx):   
        if idx < 0:
            return False
        if idx == len(self.point4solve):
            return True
        _,row,col = self.point4solve[idx]
        possible_val = []
        for v in range(9):
            if self.rows[row][v] == 0 \
                and self.cols[col][v] == 0 \
                and self.cells[self.map2cell(row,col)][v] == 0:
                possible_val.append(v)
        if len(possible_val) == 0:
            return False
        for val in possible_val:
            self.board[row][col] = str(val+1)
            self.cols[col][val] += 1
            self.rows[row][val] += 1
            self.cells[self.map2cell(row,col)][val] +=1
            if self.sudoku_solver(idx+1): # 递归 
                return True
            self.cols[col][val] -= 1
            self.rows[row][val] -= 1
            self.cells[self.map2cell(row,col)][val] -=1
            self.board[row][col] = '.'
        return False
    
    def solveSudoku(self, board):
        """
        :type board: List[List[str]]
        :rtype: bool
        """
        self.cols  = [[0]*9 for _ in range(9)]
        self.rows  = [[0]*9 for _ in range(9)]
        self.cells = [[0]*9 for _ in range(9)]
        self.board = board
        
        for row in range(9):
            for col in range(9):
                cur = self.board[row][col]
                if cur != ".":
                    cur_idx = int(cur)-1
                    self.cols[col][cur_idx] +=1
                    self.rows[row][cur_idx] +=1 
                    self.cells[self.map2cell(row,col)][cur_idx] +=1
        point_count = [(self.heat_compute(row,col),row,col) for row in range(9) for col in range(9) if self.board[row][col] == '.' ] 
        self.point4solve = sorted(point_count,reverse=True)
        self.sudoku_solver(0)


sudoku = [
  ["5","3",".",".","7",".",".",".","."],
  ["6",".",".","1","9","5",".",".","."],
  [".","9","8",".",".",".",".","6","."],
  ["8",".",".",".","6",".",".",".","3"],
  ["4",".",".","8",".","3",".",".","1"],
  ["7",".",".",".","2",".",".",".","6"],
  [".","6",".",".",".",".","2","8","."],
  [".",".",".","4","1","9",".",".","5"],
  [".",".",".",".","8",".",".","7","9"]
]

so = Solution()
so.solveSudoku(sudoku)
for line in sudoku:
    print(line)

['5', '3', '4', '6', '7', '8', '9', '1', '2']
['6', '7', '2', '1', '9', '5', '3', '4', '8']
['1', '9', '8', '3', '4', '2', '5', '6', '7']
['8', '5', '9', '7', '6', '1', '4', '2', '3']
['4', '2', '6', '8', '5', '3', '7', '9', '1']
['7', '1', '3', '9', '2', '4', '8', '5', '6']
['9', '6', '1', '5', '3', '7', '2', '8', '4']
['2', '8', '7', '4', '1', '9', '6', '3', '5']
['3', '4', '5', '2', '8', '6', '1', '7', '9']
